In [ ]:
import torch
import transformers
import peft
import datasets
import accelerate
import trl

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.path.exists("/content/train.jsonl"))

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/train.jsonl",
    split="train"
)

print(dataset)
print(dataset[0])

In [ ]:
SYSTEM_PROMPT = (
    "You are an academic search-query optimizer. Convert the user's informal "
    "research topic into a concise, keyword-rich search query for an academic "
    "paper database. Preserve every important concept from the user's original "
    "query and add useful scholarly synonyms or closely related terminology. "
    "Return ONLY the search terms. Do not write a title, question, explanation, "
    "sentence, or preamble. Do not use quotation marks. Do not include phrases "
    "such as 'scholarly search query', 'literature review', or 'research on'. "
    "Prefer approximately 6-15 useful search terms."
)


def convert_example(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }


dataset = dataset.map(
    convert_example,
    remove_columns=dataset.column_names
)

dataset = dataset.train_test_split(
    test_size=0.10,
    seed=42
)

print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["test"]))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model loaded successfully")
print("Model device:", model.device)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print(lora_config)

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/content/research-assistant-model",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    fp16=True,
    bf16=False,

    max_seq_length=256,
    packing=False,

    logging_steps=5,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=42,
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

In [ ]:
train_result = trainer.train()

In [ ]:
OUTPUT_DIR = "/content/research-assistant-model"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved fine-tuned adapter to:", OUTPUT_DIR)

In [ ]:
def test_rewrite(query):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(trainer.model.device)

    trainer.model.eval()

    with torch.no_grad():
        output = trainer.model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

In [ ]:
print(test_rewrite("AI bias in hiring algorithms"))
print(test_rewrite("sleep academic performance college"))

In [ ]:
test_queries = [
    "AI bias in hiring algorithms",
    "sleep academic performance college",
    "social media attention span studies",
    "climate change policy perspectives",
    "mental health student athletes research gaps",
]

for query in test_queries:
    print("INPUT:")
    print(query)
    print("OUTPUT:")
    print(test_rewrite(query))
    print("-" * 80)

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
print(train_result.metrics)

In [ ]:
!cd /content && zip -r research-assistant-model.zip research-assistant-model

In [ ]:
from google.colab import files

files.download("/content/research-assistant-model.zip")